## One location for the blat of all genomic sequences in design (regions + references)

In [1]:
import pandas as pd 
import yaml 

# import local module helpful functions as hf (/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/00_helpful_functions/helpful_functions.py)
import sys
sys.path.append("/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/00_helpful_functions")
import helpful_functions as hf

# # reload helpful_functions module
# import importlib
# importlib.reload(hf)


config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "../../global80K_config.yaml"
# config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [2]:
# load design file
designd_sequences = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])


### blat results from tested elements and sequences
- input for blat: /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/tested_ref_and_elements_finished.fasta (all tested sequences (regions and references))


In [ ]:
# bash commands to generate a tsv out of the psl file
# tail -n +6 blat_matched_file.psl > blat_result_without_header.tsv
file = '/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_result_without_header.tsv'
# remove header

blat_result_df = pd.read_csv(file, sep='\t', header=None)

# add column names
blat_result_columns = ['match', 'mismatch', 'rep_match', 'Ns', 'Q_gap_count', 'Q_gap_bases', 'T_gap_count', 'T_gap_bases', 'strand', 'Q_name', 'Q_size', 'Q_start', 'Q_end', 'T_name', 'T_size', 'T_start', 'T_end', 'block_count', 'blockSizes', 'qStarts', 'tStarts']

blat_result_df.columns = blat_result_columns

blat_result_df
# filter: number of matches=270, 
perfect_matches = blat_result_df[blat_result_df['match'] == 270]
perfect_matches.shape # 28276
# filter for blockSizes = 270
perfect_matches = perfect_matches[perfect_matches['blockSizes'] == '270,']
perfect_matches.shape # 28236
# check all columns for unique values and did not find any suspecious values
perfect_matches.T_name.value_counts() # 0 
# filter for matches startwith "NC_" in T_name
perfect_matches = perfect_matches[perfect_matches['T_name'].str.startswith('NC_')]
perfect_matches.shape # 27482
perfect_matches.Q_name.nunique() # 27482

# get subset of interesting columns
interesting_blat_results = ['Q_name', 'match', 'strand', 'T_name', 'T_start', 'T_end']
interesting_blat_results

perfect_matches = perfect_matches[interesting_blat_results]
perfect_matches

### Blat for intersting neuro controls
- /data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_interesting_neuro_controls_matched.tsv

In [3]:
path_to_blat_results = '/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_interesting_neuro_controls_matched.tsv'
header_names = ['match', 'mismatch', 'rep_match', 'Ns', 'Q_gap_count', 'Q_gap_bases', 'T_gap_count', 'T_gap_bases', 'strand', 'Q_name', 'Q_size', 'Q_start', 'Q_end', 'T_name', 'T_size', 'T_start', 'T_end', 'block_count', 'blockSizes', 'qStarts', 'tStarts']
blat_results = pd.read_csv(path_to_blat_results, sep="\t", header=None)
blat_results.columns = header_names
blat_results

# filter: number of matches=270, 
perfect_matches = blat_results[blat_results['match'] == 270]
perfect_matches.shape # 519
# filter for blockSizes = 270
perfect_matches = perfect_matches[perfect_matches['blockSizes'] == '270,']
perfect_matches.shape # 519
# # check all columns for unique values and did not find any suspecious values
# perfect_matches.T_name.value_counts() # 0 
# filter for matches startwith "NC_" in T_name
perfect_matches = perfect_matches[perfect_matches['T_name'].str.startswith('NC_')]
perfect_matches.shape # 500
perfect_matches.Q_name.nunique() # 498 

# get subset of interesting columns
interesting_blat_results = ['Q_name', 'match', 'strand', 'T_name', 'T_start', 'T_end']
interesting_blat_results

perfect_matches = perfect_matches[interesting_blat_results]
perfect_matches

,Q_name,match,strand,T_name,T_start,T_end
0,C_negative_neuron_MK:tile_14444_chr15_67066278...,270,+,NC_000015.10,67066277,67066547
1,C_negative_neuron_MK:tile_9599_chr12_65742272_...,270,+,NC_000012.12,65742271,65742541
2,C_negative_neuron_MK:tile_9256_chr12_26114182_...,270,+,NC_000012.12,26114181,26114451
3,C_negative_neuron_MK:tile_44145_chr8_80578146_...,270,+,NC_000008.11,80578145,80578415
6,C_negative_neuron_MK:tile_33520_chr5_77645003_...,270,+,NC_000005.10,77645002,77645272
...,...,...,...,...,...,...
4122,C_positive_neuron_NP:GW18_PFC_ABC_chr12_121536...,270,+,NC_000012.12,121536100,121536370
4123,C_positive_neuron_NP:NGN2_iPSC_ABC_chr4_850290...,270,+,NC_000004.12,85029018,85029288
4124,C_positive_neuron_NP:GW18_PFC_ABC_NGN2_iPSC_AB...,270,+,NC_000004.12,781750,782020
4125,C_positive_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,270,+,NC_000001.11,110538278,110538548


In [5]:
# check which are in the input fasta but not cannot be matched and check if you can find the sequence in the ucsc
input_fasta_intersting_controls_path = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/results/control_sequences/blat_interesting_neuro_controls.fa'
input_fasta_intersting_controls = hf.fasta_to_dataframe(input_fasta_intersting_controls_path)
input_header_set = set(input_fasta_intersting_controls['header'].to_list())
matched_sequences_set = set(perfect_matches['Q_name'].to_list())

# 

In [7]:
not_matchable_blat = input_header_set - matched_sequences_set
print(len(not_matchable_blat))
not_matchable_blat
# checked C_negative_neuron_MK:rdhs_350036_chr2_60214636_60214905_A_T_69__0.281429052959334

224


{'C_negative_neuron_MK:rdhs_350036_chr2_60214636_60214905_A_T_69__0.281429052959334',
 'C_negative_neuron_MK:rdhs_350036_chr2_60214636_60214905_C_G_257__0.366970377196711',
 'C_negative_neuron_MK:rdhs_459624_chr3_79178526_79178795_A_T_9__0.307547494437252',
 'C_negative_neuron_MK:rdhs_459624_chr3_79178526_79178795_C_G_253__0.359488132581695',
 'C_negative_neuron_MK:rdhs_581339_chr6_14500893_14501162_A_T_253__0.361955206804656',
 'C_negative_neuron_MK:rdhs_681310_chr8_76841456_76841725_T_A_141__0.3306686502775',
 'C_negative_neuron_MK:tile_10786_chr13_71759785_71760054_C_G_181__0.32568038614907',
 'C_negative_neuron_MK:tile_10789_chr13_71760386_71760655_C_G_193__0.288427385780119',
 'C_negative_neuron_MK:tile_11200_chr13_105364096_105364365_A_T_201__0.290852855921551',
 'C_negative_neuron_MK:tile_11200_chr13_105364096_105364365_T_A_13__0.27122115537836',
 'C_negative_neuron_MK:tile_13042_chr14_103542966_103543235_G_C_129__0.385465789555967',
 'C_negative_neuron_MK:tile_13042_chr14_10354

### Blat for control variants (tried to do all REF sequences)
- problem: only 309 perfekt matches
- what is the expected number of sequences in the control we want a region for? ~ 6K
- make new blat investigation with new headers

In [ ]:
# input for the controls: (should be all the ref and regions from the variant control set)
input_fasta = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/results/control_sequences/control_variant_reference_sequence.fasta'

In [8]:
# load blat results and check if only one perfect match of each sequence
header_names = ['match', 'mismatch', 'rep_match', 'Ns', 'Q_gap_count', 'Q_gap_bases', 'T_gap_count', 'T_gap_bases', 'strand', 'Q_name', 'Q_size', 'Q_start', 'Q_end', 'T_name', 'T_size', 'T_start', 'T_end', 'block_count', 'blockSizes', 'qStarts', 'tStarts']
path_to_blat_results = '/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_variant_control_sequences_matched_file.tsv'
blat_results = pd.read_csv(path_to_blat_results, sep="\t", header=None)
blat_results.columns = header_names
blat_results

# filter: number of matches=270, 
perfect_matches = blat_results[blat_results['match'] == 270]
perfect_matches.shape # 314
# filter for blockSizes = 270
perfect_matches = perfect_matches[perfect_matches['blockSizes'] == '270,']
perfect_matches.shape # 311
# check all columns for unique values and did not find any suspecious values
perfect_matches.T_name.value_counts() # 0 
# filter for matches startwith "NC_" in T_name
perfect_matches = perfect_matches[perfect_matches['T_name'].str.startswith('NC_')]
perfect_matches.shape # 309
perfect_matches.Q_name.nunique() # 309

# get subset of interesting columns
interesting_blat_results = ['Q_name', 'match', 'strand', 'T_name', 'T_start', 'T_end']
interesting_blat_results

perfect_matches = perfect_matches[interesting_blat_results]
perfect_matches

,Q_name,match,strand,T_name,T_start,T_end
0,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154813248,154813518
1,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154839744,154840014
2,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154840018,154840288
3,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154840331,154840601
4,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,270,+,NC_000001.11,154860467,154860737
...,...,...,...,...,...,...
5563,C_positive_heart_CAD:REF_rs7865618,270,+,NC_000009.12,22030870,22031140
5939,C_positive_heart_CAD:REF_rs4977757,270,+,NC_000009.12,22094195,22094465
6625,C_positive_heart_CAD:REF_rs1537373,270,+,NC_000009.12,22103206,22103476
6626,C_positive_heart_CAD:REF_rs10811656,270,+,NC_000009.12,22124337,22124607


In [11]:
perfect_matches['labels'] = perfect_matches['Q_name'].apply(hf.get_label)

In [10]:
labels.value_counts()

Q_name
GC_Selvarajan            165
C_positive_heart_CAD      48
GC_Mendelian_variants     46
GC_Atrial_fib             21
GC_Mohlke                 16
GC_Liang                   8
GC_Kircher                 5
Name: count, dtype: int64

In [12]:
perfect_matches[perfect_matches['labels'] == 'GC_Selvarajan']

,Q_name,match,strand,T_name,T_start,T_end,labels
218,GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs...,270,+,NC_000001.11,3036131,3036401,GC_Selvarajan
219,GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs...,270,+,NC_000001.11,3039882,3040152,GC_Selvarajan
220,GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs...,270,+,NC_000001.11,3039924,3040194,GC_Selvarajan
221,GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs...,270,+,NC_000001.11,3039966,3040236,GC_Selvarajan
222,GC_Selvarajan:REF_rs2493288|STARR-seq-HepG2_fw...,270,+,NC_000001.11,3414235,3414505,GC_Selvarajan
...,...,...,...,...,...,...,...
5143,GC_Selvarajan:REF_rs10121140|STARR-seq-HepG2~r...,270,+,NC_000009.12,14292520,14292790,GC_Selvarajan
5144,GC_Selvarajan:REF_rs10121140|STARR-seq-HepG2~r...,270,+,NC_000009.12,14292585,14292855,GC_Selvarajan
5145,GC_Selvarajan:REF_rs6475604|STARR-seq-HepG2_fw...,270,+,NC_000009.12,22052630,22052900,GC_Selvarajan
5146,GC_Selvarajan:REF_rs1333045|STARR-seq-HepG2_fw...,270,+,NC_000009.12,22119060,22119330,GC_Selvarajan
